In [ ]:
from torch_uncertainty.metrics import AURC

import sys
sys.path.append("../scripts")

import numpy as np
from pathlib import Path
import torch
from scipy.special import log_softmax

from callm.data.classification import DATASETS


%load_ext autoreload
%autoreload 2


/home/estienne/anaconda3/envs/callm/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
class AUROCMetric(AURC):

    def update(self, confidences, correctness):
        self.scores.append(confidences.float())
        self.errors.append((1 - correctness).float())


aurc = AUROCMetric()
for dataset_model in DATASETS:
    data_path = Path("../scores/classification") / dataset_model
    logprobs = torch.from_numpy(log_softmax(np.load(data_path / "scores.npy"), axis=1)).float()
    labels = torch.from_numpy(np.load(data_path / "targets.npy").reshape(-1))
    confidences = torch.exp(logprobs).max(dim=1).values
    correctness = (logprobs.argmax(dim=1) == labels).float()
    aurc.update(confidences, correctness)
    aurc_score = aurc.compute()
    aurc.reset()
    # shifted_confidences = confidences + 0.5
    shifted_confidences = torch.maximum(torch.tensor(1.0), confidences + 0.8)
    aurc.update(shifted_confidences, correctness)
    shifted_aurc_score = aurc.compute()
    aurc.reset()
    print(f"{dataset_model} - AURC: {aurc_score:.4f}, Shifted AURC: {shifted_aurc_score:.4f}, diff: {(shifted_aurc_score - aurc_score) if (shifted_aurc_score - aurc_score) > 1e-3 else '< 1e-3'}")


sst2_gpt2_4shot - AURC: 0.1783, Shifted AURC: 0.1783, diff: < 1e-3
sst2_gpt2 - AURC: 0.1821, Shifted AURC: 0.1821, diff: < 1e-3
sitw_plda - AURC: 0.0000, Shifted AURC: 0.0000, diff: < 1e-3
fvcaus_plda - AURC: 0.0095, Shifted AURC: 0.0095, diff: < 1e-3
cifar100_repvgg_a2 - AURC: 0.0611, Shifted AURC: 0.0611, diff: < 1e-3
cifar100_resnet-20 - AURC: 0.1115, Shifted AURC: 0.1115, diff: < 1e-3
cifar100_vgg19_bn - AURC: 0.0824, Shifted AURC: 0.0824, diff: < 1e-3
pneumoniamnist_resnet50 - AURC: 0.0312, Shifted AURC: 0.0316, diff: < 1e-3
adrenalmnist_resnet50 - AURC: 0.0796, Shifted AURC: 0.0796, diff: < 1e-3
pathmnist_resnet50 - AURC: 0.0210, Shifted AURC: 0.0240, diff: 0.0029727090150117874
iemocap_wav2vec_pt - AURC: 0.2085, Shifted AURC: 0.2085, diff: < 1e-3
agnews_gpt2 - AURC: 0.4352, Shifted AURC: 0.4352, diff: < 1e-3
cifar10_resnet-20 - AURC: 0.0092, Shifted AURC: 0.0092, diff: < 1e-3
cifar10_vgg19_bn - AURC: 0.0075, Shifted AURC: 0.0075, diff: < 1e-3
cifar10_repvgg_a2 - AURC: 0.0061, Sh